# TCN Training Only (Clean)

This notebook is for **training only**.
It uses isolated `train_*` variables and a Sharpe-based checkpoint policy.

## 1) Connect to Colab VM and Sync Repo
Run this first.

In [1]:
import gc, shutil, subprocess, sys
from pathlib import Path

TRAIN_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
TRAIN_REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")
TRAIN_BRANCH = "feature/experimental-updates-20260302"  # <- your branch

if not (TRAIN_REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", TRAIN_REPO_URL, str(TRAIN_REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(TRAIN_REPO_DIR), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(TRAIN_REPO_DIR), "checkout", TRAIN_BRANCH], check=True)
subprocess.run(["git", "-C", str(TRAIN_REPO_DIR), "reset", "--hard", f"origin/{TRAIN_BRANCH}"], check=True)

purge_paths = [
    TRAIN_REPO_DIR / "tcn_fusion_results",
    TRAIN_REPO_DIR / "tcn_results",
    TRAIN_REPO_DIR / "tcn_att_results",
    TRAIN_REPO_DIR / "output_log",   # singular
    TRAIN_REPO_DIR / "output_logs",  # keep both just in case
    TRAIN_REPO_DIR / "data" / "phase1_preparation_artifacts",
    TRAIN_REPO_DIR / "data" / "master_features_NORMALIZED.csv",
    TRAIN_REPO_DIR / "data" / "daily_ohlcv_assets.csv",
    TRAIN_REPO_DIR / "data" / "processed_daily_macro_features.csv",
]

deleted = []
for p in purge_paths:
    if p.is_dir():
        shutil.rmtree(p, ignore_errors=True); deleted.append(str(p))
    elif p.is_file():
        p.unlink(missing_ok=True); deleted.append(str(p))

for cache_dir in TRAIN_REPO_DIR.rglob("__pycache__"):
    shutil.rmtree(cache_dir, ignore_errors=True)
for ckpt_dir in TRAIN_REPO_DIR.rglob(".ipynb_checkpoints"):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules.keys()):
    if mod.startswith("src.") or mod.startswith("src_"):
        del sys.modules[mod]
gc.collect()

print("✅ Fresh start complete on branch:", TRAIN_BRANCH)

✅ Fresh start complete on branch: feature/experimental-updates-20260302


In [2]:
from pathlib import Path
import os

root = Path(globals().get("TRAIN_REPO_DIR", "/content/tcn_tape_vectorized_version_clean"))
print("Exists:", root.exists())
print("CWD:", os.getcwd())

print("\nTop-level:")
for p in sorted(root.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f" - [{kind}] {p.name}")

# Quick check for outputs/caches you expected to be deleted
targets = [
    "tcn_fusion_results",
    "tcn_results",
    "tcn_att_results",
    "output_log",
    "output_logs",
    "data/phase1_preparation_artifacts",
    "data/master_features_NORMALIZED.csv",
    "data/daily_ohlcv_assets.csv",
    "data/processed_daily_macro_features.csv",
]
print("\nTarget paths:")
for t in targets:
    p = root / t
    print(f" - {t}: {'EXISTS' if p.exists() else 'MISSING'}")


Exists: True
CWD: /content

Top-level:
 - [DIR ] .git
 - [FILE] .gitignore
 - [FILE] RL Portfolio Optimization Feature Engineering.md
 - [FILE] RL_Portfolio_Optimization_Feature_Engineering.ipynb
 - [FILE] USAGE_GUIDE_ACTUARIAL.py
 - [FILE] __init__.py
 - [FILE] convert_md_to_ipynb.py
 - [DIR ] data
 - [FILE] debug_attention_weights.py
 - [DIR ] docs
 - [DIR ] eval
 - [DIR ] paper
 - [DIR ] prompts
 - [FILE] ra_kl_research_writeup.ipynb
 - [FILE] rcdcc_research_writeup.ipynb
 - [FILE] requirements.txt
 - [FILE] run_tcn_eval.py
 - [DIR ] src
 - [FILE] tcn_architecture_analysis.ipynb
 - [FILE] tcn_architecture_analysis_dual_head_redundancy.ipynb
 - [DIR ] tcn_documentation
 - [FILE] tcn_evaluation_only.ipynb
 - [FILE] technical_deep_dive_presentation.ipynb
 - [DIR ] tests
 - [FILE] traditional_portfolio_benchmarks.ipynb
 - [DIR ] training_scripts

Target paths:
 - tcn_fusion_results: MISSING
 - tcn_results: MISSING
 - tcn_att_results: MISSING
 - output_log: MISSING
 - output_logs: MISSIN

In [3]:
#!find /content/tcn_tape_vectorized_version_clean -maxdepth 3 | head -n 300

In [4]:
# Install project requirements in Colab VM
#import subprocess, sys
#from pathlib import Path

REPO_DIR = Path(globals().get("TRAIN_REPO_DIR", "/content/tcn_tape_vectorized_version_clean"))
REQ_FILE = REPO_DIR / "requirements.txt"

if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

print("Using python:", sys.executable)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)], check=True)

print("✅ Requirements installed")


Using python: /usr/bin/python3
✅ Requirements installed


In [5]:
# --- GPU sanity/setup for TensorFlow ---
import tensorflow as tf

!nvidia-smi -L

gpus = tf.config.list_physical_devices("GPU")
print("TF GPUs:", gpus)
if not gpus:
    raise RuntimeError("No GPU visible to TensorFlow. In Colab: Runtime -> Change runtime type -> GPU")

for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

USE_MIXED_PRECISION = False  # set True only after stable fp32 training
if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
else:
    tf.keras.mixed_precision.set_global_policy("float32")
print("Mixed precision policy:", tf.keras.mixed_precision.global_policy())

with tf.device("/GPU:0"):
    a = tf.random.normal((4096, 4096))
    b = tf.random.normal((4096, 4096))
    c = tf.matmul(a, b)

print("Matmul device:", c.device)
print("Default GPU device name:", tf.test.gpu_device_name())

GPU 0: NVIDIA H100 80GB HBM3 (UUID: GPU-6c69c5a8-6de3-c1d0-dad6-db1f05484974)
TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision policy: <DTypePolicy "float32">
Matmul device: /job:localhost/replica:0/task:0/device:GPU:0
Default GPU device name: /device:GPU:0


## 2) Imports

In [6]:
import os, sys
from pathlib import Path

REPO_DIR = Path(globals().get("TRAIN_REPO_DIR", "/content/tcn_tape_vectorized_version_clean"))

if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repo not found: {REPO_DIR}")

# Set working directory
os.chdir(REPO_DIR)

# Add repo root to Python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

cwd: /content/tcn_tape_vectorized_version_clean
sys.path[0]: /content/tcn_tape_vectorized_version_clean


In [7]:
from copy import deepcopy
from pathlib import Path

import pandas as pd

from src.config import get_active_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import prepare_phase1_dataset, run_experiment6_tape

## 3) Base Config and Dataset Prep

In [8]:
# ------------------------------------------------------------------
# Global feature-audit plan enforcement (49 + 4 actuarial = 53)
# ------------------------------------------------------------------

def enforce_feature_audit_plan(cfg):
    fs = cfg.setdefault("feature_params", {}).setdefault("feature_selection", {})
    fs["enforce_allowlist"] = True
    fs["allowlist_apply_to_phase2"] = False

    allowlist = list(dict.fromkeys(fs.get("active_features_allowlist", []) or []))
    fs["active_features_allowlist"] = allowlist

    plan_name = fs.get("feature_audit_plan_name", "feature_audit_allowlist")
    expected_total = int(fs.get("feature_audit_expected_total_count", len(allowlist)))
    act_cols = [c for c in allowlist if str(c).startswith("Actuarial_")]

    print("✅ Feature audit plan configured")
    print("   plan:", plan_name)
    print("   allowlist count:", len(allowlist))
    print("   expected total:", expected_total)
    print("   actuarial in allowlist:", len(act_cols), act_cols)

    if len(allowlist) != expected_total:
        print("⚠️ Allowlist count differs from expected total. Check src/config.py")

    return cfg


In [9]:
from pathlib import Path
TRAIN_RANDOM_SEED = 42

train_config = deepcopy(get_active_config("phase1"))

# Optional: override analysis horizon
# train_config["ANALYSIS_END_DATE"] = "2025-09-01"

train_config = enforce_feature_audit_plan(train_config)

# Force fresh dataset build and market data re-download
if "train_phase1_data" in globals():
    del train_phase1_data

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(Path(globals().get("TRAIN_REPO_DIR", "/content/tcn_tape_vectorized_version_clean")) / "data_exports"),
)


✅ Feature audit plan configured
   plan: exp6_feature_audit_20260221_v2
   allowlist count: 59
   expected total: 59
   actuarial in allowlist: 4 ['Actuarial_Expected_Recovery', 'Actuarial_Prob_30d', 'Actuarial_Prob_60d', 'Actuarial_Reserve_Severity']
📊 Loading raw market data...
   ✅ Raw data shape: (55107, 7)
   ✅ Date range: 2003-09-02 00:00:00 → 2025-08-29 00:00:00

🔧 Computing multi-horizon log returns: [1, 5, 10, 21]
   ✅ Shape after returns: (54897, 11)

📈 Calculating 21-day rolling statistics

🧮 Computing technical indicators

🕯️ Adding candlestick geometry features (if enabled)

📊 Computing dynamic covariance features

🎯 Adding regime awareness features
   ✅ Master DF shape: (54767, 38)
   ✅ Total features: 38

📊 Integrating fundamental features (if enabled)...
   ✅ Fundamental columns in dataset: 0 (enabled=False)

📊 Integrating macroeconomic features (if enabled)...
   ✅ Macro features added - 12 columns: ['SOFR_diff', 'DGS10_level', 'DGS10_diff', 'T10Y2Y_level', 'TIPS10Y_le


📊 Integrating actuarial features (if enabled)...


   ✅ Actuarial columns in dataset: 4 (enabled=True)
   📋 Non-null counts: {'Actuarial_Expected_Recovery': 54767, 'Actuarial_Prob_30d': 54767, 'Actuarial_Prob_60d': 54767, 'Actuarial_Reserve_Severity': 54767}

✅ Final master DF shape: (54767, 66)
   ✅ Total features: 66
🧭 Feature audit plan: exp6_feature_audit_20260221_v2 (allowlist enabled)
   active feature count (phase1): 59
   ✅ expected active features: 59

✂️ FILTERING TO ANALYSIS PERIOD
   Filtering data to: 2003-09-02 → 2025-09-01
   ✅ Dates after filter: 5501 trading days
   ✅ Date range: 2003-10-20 00:00:00 to 2025-08-29 00:00:00
✂️  TIME-BASED TRAIN/TEST SPLIT (80/20 split)
   Train: 2003-10-20 → 2021-04-13 (4400 days, 17.5 years, 43757 rows)
   Test:  2021-04-14 → 2025-08-29 (1101 days, 4.4 years, 11010 rows)

🔧 NORMALISING FEATURES (standard scaler)

💾 Saving NORMALISED master dataframe to '/content/tcn_tape_vectorized_version_clean/data/master_features_NORMALIZED.csv'

💾 Saved preparation artifacts:
   raw OHLCV: /content/

In [10]:
print("Train shape:", train_phase1_data.train_df.shape)
print("Test shape:", train_phase1_data.test_df.shape)

cols = train_phase1_data.train_df.columns
print("Total columns:", len(cols))

# quick sanity for common redundant groups
dup_like = [c for c in cols if c.endswith("_raw") or c.endswith("_unscaled")]
print("Potential redundant raw/unscaled cols:", len(dup_like))
print(dup_like[:20])

used_now = list(dict.fromkeys(train_phase1_data.data_processor.get_feature_columns("phase1")))
act_now = [c for c in used_now if c.startswith("Actuarial_")]
print("Model feature count (phase1):", len(used_now))
print("Actuarial feature count:", len(act_now), act_now)


Train shape: (43757, 66)
Test shape: (11010, 66)
Total columns: 66
Potential redundant raw/unscaled cols: 0
[]
Model feature count (phase1): 59
Actuarial feature count: 4 ['Actuarial_Expected_Recovery', 'Actuarial_Prob_30d', 'Actuarial_Prob_60d', 'Actuarial_Reserve_Severity']


In [ ]:
[i for i in cols]

In [11]:
train_phase1_data.train_df[['Actuarial_Prob_60d', 'Actuarial_Prob_30d']].head()

,Actuarial_Prob_60d,Actuarial_Prob_30d
0,1.0,1.0
1,1.0,1.0
2,1.0,1.0
3,1.0,1.0
4,1.0,1.0


In [12]:
used = set(train_phase1_data.data_processor.get_feature_columns("phase1"))
disabled = set(train_config["feature_params"]["feature_selection"]["disabled_features"])
act_used = sorted([c for c in used if c.startswith("Actuarial_")])

print("Used feature count:", len(used))
print("Actuarial used:", len(act_used), act_used)
print("Disabled that still in used:", sorted(disabled & used))  # should be []
print("VIX_zscore used?", "VIX_zscore" in used)


Used feature count: 59
Actuarial used: 4 ['Actuarial_Expected_Recovery', 'Actuarial_Prob_30d', 'Actuarial_Prob_60d', 'Actuarial_Reserve_Severity']
Disabled that still in used: []
VIX_zscore used? True


In [13]:
base_cols = ["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]
keep = [c for c in base_cols + list(used) if c in train_phase1_data.master_df.columns]

train_phase1_data.master_df = train_phase1_data.master_df[keep].copy()
train_phase1_data.train_df = train_phase1_data.train_df[keep].copy()
train_phase1_data.test_df  = train_phase1_data.test_df[keep].copy()

## 4) Training Overrides (Performance-Fixed v2)

Applies all 8 performance fixes from the deep-dive analysis:
PPO stability, smooth schedules, EMA reward norm, alpha diversity,
extended training budget (500K), execution inertia, EMA actor,
and architecture simplification.

**Note:** The config in `src/config.py` already contains the base fixes.
These overrides ensure the notebook is fully aligned.

In [14]:
# ============================================================================
# PERFORMANCE-FIXED OVERRIDES (v2 — aligned with all 8 state-of-art fixes)
# ============================================================================
# Fixes applied:
#   #1 PPO Stability: tighter clip, relaxed KL, lower LR
#   #2 Smooth Schedules: linear interpolation, wider spacing for 500K
#   #3 EMA Reward Normalization: handled in ppo_agent_tf.py
#   #4 Alpha Diversity: exp_tanh activation, temp=0.5, HHI aux loss
#   #5 Training Budget: 500K timesteps
#   #6 Execution Inertia: beta schedule 0.50→0.25
#   #7 EMA Actor: Polyak-averaged weights (handled in ppo_agent_tf.py)
#   #8 Architecture Simplification: disable undertrained components
# ============================================================================
from copy import deepcopy

train_config = deepcopy(train_config)

tp = train_config["training_params"]
ap = train_config["agent_params"]
ppo = ap["ppo_params"]
env = train_config["environment_params"]
fp = train_config.setdefault("feature_params", {})
fund_cfg = fp.setdefault("fundamental_features", {})
act_cfg = fp.setdefault("actuarial_params", {})

# Hard requirement: no fundamentals, actuarial ON
fund_cfg["enabled"] = False
act_cfg["enabled"] = True

# ---- Fix #5: Extended Training Budget ----
tp["max_total_timesteps"] = 500_000
tp["timesteps_per_ppo_update"] = 1008
tp["num_parallel_envs"] = 4

tp["timesteps_per_ppo_update_schedule"] = [
    {"threshold": 0, "timesteps_per_update": 1008},
    {"threshold": 150_000, "timesteps_per_update": 1512},
    {"threshold": 300_000, "timesteps_per_update": 2016},
]

tp["batch_size_ppo_schedule"] = [
    {"threshold": 0, "batch_size": 252},
    {"threshold": 150_000, "batch_size": 336},
    {"threshold": 300_000, "batch_size": 504},
]

# ---- A2: TCN backbone (unchanged) ----
ap["tcn_filters"] = [64, 96, 128, 128, 128]
ap["tcn_kernel_size"] = 5
ap["tcn_dilations"] = [1, 2, 4, 8, 16]
ap["tcn_dropout"] = 0.15

# ---- Fix #1: PPO Stability ----
ppo["num_ppo_epochs"] = 3
ppo["policy_clip"] = 0.15           # tighter than 0.2
ppo["target_kl"] = 0.0             # re-enabled with conservative limit
ppo["kl_stop_multiplier"] = 2.0     # wider headroom
ppo["minibatches_before_kl_stop"] = 2
ppo["max_grad_norm"] = 0.50
ppo["value_clip"] = 0.3             # wider value clipping

ppo["actor_lr"] = 3e-5              # slightly higher for 500K
ppo["critic_lr"] = 1.5e-4
ppo["entropy_coef"] = 0.005         # reduced entropy for stability

# ---- Fix #4b: Alpha Diversity HHI auxiliary ----
ppo["alpha_diversity_coef"] = 0.01

# Risk-aware actor auxiliaries
ppo["use_risk_aux_loss"] = True
ppo["risk_aux_return_feature_index"] = 0
ppo["risk_aux_cash_return"] = 0.0
ppo["risk_aux_sharpe_coef"] = 0.0
ppo["risk_aux_mvo_coef"] = 0.0
ppo["risk_aux_cvar_coef"] = 0.002
ppo["risk_aux_cvar_alpha"] = 0.05
ppo["risk_aux_mvo_cov_ridge"] = 1e-3
ppo["risk_aux_mvo_long_only"] = True
ppo["risk_aux_mvo_risky_budget"] = 0.95

# ---- Fix #2: Smooth schedules (scaled for 500K) ----
tp["actor_lr_schedule"] = [
    {"threshold": 0, "lr": 3e-5},
    {"threshold": 150_000, "lr": 2e-5},
    {"threshold": 350_000, "lr": 1e-5},
]

tp["ppo_gamma_schedule"] = [
    {"threshold": 0, "gamma": 0.990},
    {"threshold": 150_000, "gamma": 0.995},
    {"threshold": 350_000, "gamma": 0.998},
]
tp["ppo_gae_lambda_schedule"] = [
    {"threshold": 0, "gae_lambda": 0.92},
    {"threshold": 150_000, "gae_lambda": 0.95},
    {"threshold": 350_000, "gae_lambda": 0.97},
]

# RA-KL disabled (re-enable after baseline improves)
tp["ra_kl_enabled"] = False
tp["ra_kl_target_ratio"] = 1.0
tp["ra_kl_ema_alpha"] = 0.25
tp["ra_kl_gain"] = 0.03
tp["ra_kl_deadband"] = 0.20
tp["ra_kl_max_change_fraction"] = 0.05
tp["ra_kl_min_target_kl"] = 0.016
tp["ra_kl_max_target_kl"] = 0.030

# ---- Fix #4a/c: Dirichlet + concentration ----
ap["dirichlet_alpha_activation"] = "exp_tanh"  # bounded, diversity-promoting
ap["dirichlet_logit_temperature"] = 0.5          # sharpen pre-activation
ap["dirichlet_alpha_cap"] = 50.0                 # relaxed cap
ap["dirichlet_epsilon"] = {"max": 0.2, "min": 0.02}

ap["dirichlet_adaptive_temperature_enabled"] = False  # not needed with exp_tanh

# ---- Fix #8: Architecture Simplification ----
# Keep mixer but reduce layers; disable undertrained components
ap["fusion_cross_asset_mixer_enabled"] = True
ap["fusion_cross_asset_mixer_layers"] = 1        # reduced from 2
ap["fusion_cross_asset_mixer_expansion"] = 2.0
ap["fusion_cross_asset_mixer_dropout"] = 0.10
ap["fusion_asset_identity_enabled"] = True
ap["fusion_context_cross_attention_enabled"] = False  # DISABLED
ap["fusion_per_asset_alpha_head"] = True
ap["fusion_alpha_head_hidden_dims"] = [128, 64]
ap["fusion_alpha_head_dropout"] = 0.05

ap["recurrent_memory_enabled"] = False           # DISABLED
ap["regime_conditioning_enabled"] = False         # DISABLED
ap["state_augmentation_enabled"] = False
ap["distributional_critic_enabled"] = False       # DISABLED

env["concentration_penalty_scalar"] = 3.0
env["concentration_target_hhi"] = 0.12
env["top_weight_penalty_scalar"] = 2.0
env["action_realization_penalty_scalar"] = 0.5

# ---- Fix #6: Execution inertia + turnover (smooth schedules) ----
env["target_turnover"] = 0.35
env["turnover_penalty_scalar"] = 0.05
env["transaction_cost_pct"] = 0.001

tp["action_execution_beta_schedule"] = [
    {"threshold": 0, "beta": 0.50},
    {"threshold": 100_000, "beta": 0.40},
    {"threshold": 200_000, "beta": 0.30},
    {"threshold": 350_000, "beta": 0.25},
]
tp["evaluation_action_execution_beta"] = 0.25

# Also set dict-format for tcn_phase1.py initial env creation
tp["action_execution_beta_curriculum"] = {
    0: 0.50,
    100_000: 0.40,
    200_000: 0.30,
    350_000: 0.25,
}

tp["turnover_penalty_curriculum"] = {
    0: 0.75,
    30_000: 1.25,
    60_000: 1.50,
    90_000: 1.75,
    120_000: 2.0,
}
tp["evaluation_turnover_penalty_scalar"] = 2.0

# ---- Fix #2: Episode horizon curriculum (scaled for 500K) ----
tp["use_episode_length_curriculum"] = True
tp["episode_length_curriculum_schedule"] = [
    {"threshold": 0, "limit": 756},
    {"threshold": 100_000, "limit": 1008},
    {"threshold": 250_000, "limit": 1500},
    {"threshold": 400_000, "limit": None},  # uncapped
]

# ---- Logging + checkpoints ----
tp["log_step_diagnostics"] = True
tp["update_log_interval"] = 10
tp["alpha_diversity_log_interval"] = 2
tp["alpha_diversity_warning_after_updates"] = 10
tp["alpha_diversity_warning_std_threshold"] = 0.25

tp["deterministic_validation_checkpointing_enabled"] = False
tp["deterministic_validation_eval_every_episodes"] = 5
tp["deterministic_validation_mode"] = "mean"
tp["deterministic_validation_episode_length_limit"] = None
tp["deterministic_validation_episode_length_limit_curriculum"] = [
    {"threshold": 0, "limit": 756},
    {"threshold": 100_000, "limit": 1008},
    {"threshold": 250_000, "limit": 1500},
    {"threshold": 400_000, "limit": None},
]
tp["deterministic_validation_sharpe_min"] = 0.5
tp["deterministic_validation_sharpe_min_delta"] = 0.005
tp["deterministic_validation_seed_offset"] = 10_000
tp["deterministic_validation_log_alpha_stats"] = True
tp["deterministic_validation_checkpointing_only"] = False

tp["high_watermark_checkpoint_enabled"] = True
tp["high_watermark_sharpe_threshold"] = 0.7
tp["high_watermark_max_drawdown_abs_threshold"] = 0.25
tp["high_watermark_skip_on_deterministic_validation_trigger"] = True
tp["step_sharpe_checkpoint_enabled"] = False
tp["periodic_checkpoint_every_steps"] = 0
tp["rare_checkpoint_params"] = {"enable": False}
tp["tape_checkpoint_threshold"] = 999.0

# PopArt + multi-horizon reward decomposition (keep enabled)
ppo["popart_enabled"] = True
ppo["popart_min_std"] = 1e-3
ppo["multi_horizon_reward_enabled"] = True
ppo["multi_horizon_reward_coef"] = 0.20
ppo["multi_horizon_reward_horizons"] = [21, 63, 126, 252]
ppo["multi_horizon_reward_weights"] = [0.15, 0.25, 0.30, 0.30]

ppo["risk_aux_cvar_adaptive_enabled"] = True
ppo["risk_aux_cvar_target"] = 0.015
ppo["risk_aux_cvar_adapt_lr"] = 0.05
ppo["risk_aux_cvar_min_coef"] = 0.0
ppo["risk_aux_cvar_max_coef"] = 0.06

tp["episode_length_curriculum_smooth_enabled"] = True
tp["episode_length_curriculum_overlap_steps"] = 10_000

tp["deterministic_validation_multi_horizon_enabled"] = False
tp["deterministic_validation_multi_horizon_limits"] = [252, 504, 756, 1008]
tp["deterministic_validation_multi_horizon_weights"] = [0.35, 0.30, 0.20, 0.15]
tp["deterministic_validation_multi_horizon_dd_penalty_coef"] = 0.25
tp["deterministic_validation_stochastic_sanity_enabled"] = False
tp["deterministic_validation_stochastic_sanity_runs"] = 3
tp["deterministic_validation_stochastic_sanity_episode_length_limit"] = 252
tp["deterministic_validation_stochastic_sanity_min_mean_sharpe"] = 0.0
tp["deterministic_validation_stochastic_sanity_max_sharpe_std"] = 1.5

# ---- Print Summary ----
print("\u2705 Applied PERFORMANCE-FIXED overrides (v2)")
print(f"  Fix #1 PPO: clip={ppo['policy_clip']}, epochs={ppo['num_ppo_epochs']}, target_kl={ppo['target_kl']}, kl_stop={ppo['kl_stop_multiplier']}, entropy={ppo['entropy_coef']}")
print(f"  Fix #2 Schedules: gamma {tp['ppo_gamma_schedule'][-1]['gamma']}@{tp['ppo_gamma_schedule'][-1]['threshold']:,}, gae {tp['ppo_gae_lambda_schedule'][-1]['gae_lambda']}@{tp['ppo_gae_lambda_schedule'][-1]['threshold']:,}")
print(f"  Fix #4 Alpha: activation={ap['dirichlet_alpha_activation']}, temp={ap['dirichlet_logit_temperature']}, cap={ap['dirichlet_alpha_cap']}, HHI_coef={ppo['alpha_diversity_coef']}")
print(f"  Fix #5 Budget: max_timesteps={tp['max_total_timesteps']:,}")
print(f"  Fix #6 Beta schedule: {tp['action_execution_beta_schedule']}")
print(f"  Fix #8 Simplified: recurrent={ap['recurrent_memory_enabled']}, distributional={ap['distributional_critic_enabled']}, cross_attn={ap['fusion_context_cross_attention_enabled']}, mixer_layers={ap['fusion_cross_asset_mixer_layers']}")
print(f"  Episode curriculum: {tp['episode_length_curriculum_schedule']}")
print(f"  Turnover curriculum: {tp['turnover_penalty_curriculum']}")


✅ Applied PERFORMANCE-FIXED overrides (v2)
  Fix #1 PPO: clip=0.15, epochs=3, target_kl=0.02, kl_stop=2.0, entropy=0.005
  Fix #2 Schedules: gamma 0.998@350,000, gae 0.97@350,000
  Fix #4 Alpha: activation=exp_tanh, temp=0.5, cap=50.0, HHI_coef=0.01
  Fix #5 Budget: max_timesteps=500,000
  Fix #6 Beta schedule: [{'threshold': 0, 'beta': 0.5}, {'threshold': 100000, 'beta': 0.4}, {'threshold': 200000, 'beta': 0.3}, {'threshold': 350000, 'beta': 0.25}]
  Fix #8 Simplified: recurrent=False, distributional=False, cross_attn=False, mixer_layers=1
  Episode curriculum: [{'threshold': 0, 'limit': 756}, {'threshold': 100000, 'limit': 1008}, {'threshold': 250000, 'limit': 1500}, {'threshold': 400000, 'limit': None}]
  Turnover curriculum: {0: 0.75, 30000: 1.25, 60000: 1.5, 90000: 1.75, 120000: 2.0}


In [15]:
# Optional experimental override (OFF by default)
# Keep this OFF for the aligned default pipeline.
EXPERIMENT_DISABLE_KL_GUARDS = False

if EXPERIMENT_DISABLE_KL_GUARDS:
    tp = train_config["training_params"]
    ppo = train_config["agent_params"]["ppo_params"]

    # Disable RA-KL controller (otherwise it keeps adjusting target_kl)
    tp["ra_kl_enabled"] = False

    # Disable KL early-stop gate in PPOAgentTF
    ppo["target_kl"] = 0.0

    # Optional (irrelevant once target_kl=0, but explicit)
    ppo["kl_stop_multiplier"] = 999.0
    ppo["minibatches_before_kl_stop"] = 9999
    print("⚠️ EXPERIMENT_DISABLE_KL_GUARDS=True (non-default experimental mode)")
else:
    print("ℹ️ EXPERIMENT_DISABLE_KL_GUARDS=False (keeping RA-KL + KL safeguards)")



ℹ️ EXPERIMENT_DISABLE_KL_GUARDS=False (keeping RA-KL + KL safeguards)


## 5) Run Training

In [16]:
RUN_TRAINING = True

if RUN_TRAINING:
    tp = train_config["training_params"]
    print("🚀 Starting training")
    print("Architecture:", train_config["agent_params"].get("actor_critic_type"))
    print("max_total_timesteps:", tp["max_total_timesteps"])
    print("num_parallel_envs:", tp.get("num_parallel_envs", 1))

    actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith("Actuarial_")]
    if not actuarial_cols:
        raise RuntimeError("Actuarial features missing in train_phase1_data.master_df")
    actuarial_non_null = {c: int(train_phase1_data.master_df[c].notna().sum()) for c in actuarial_cols}
    if any(v == 0 for v in actuarial_non_null.values()):
        raise RuntimeError(f"Actuarial features present but empty: {actuarial_non_null}")

    fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith("Fundamental_")]
    if fundamental_cols:
        raise RuntimeError(f"Fundamental columns still present (expected removed): {fundamental_cols}")

    print("✅ Actuarial feature check passed:", actuarial_non_null)
    print("✅ Fundamental feature check passed: none present")

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config["agent_params"].get("actor_critic_type"),
        timesteps_per_update=tp.get("timesteps_per_ppo_update", 384),
        max_total_timesteps=tp["max_total_timesteps"],
    )

    print("✅ Training complete")
    print("checkpoint_prefix:", train_experiment6.checkpoint_path)
else:
    print("ℹ️ RUN_TRAINING=False")

🚀 Starting training
Architecture: TCN_FUSION
max_total_timesteps: 500000
num_parallel_envs: 4
✅ Actuarial feature check passed: {'Actuarial_Prob_30d': 54767, 'Actuarial_Reserve_Severity': 54767, 'Actuarial_Expected_Recovery': 54767, 'Actuarial_Prob_60d': 54767}
✅ Fundamental feature check passed: none present

EXPERIMENT 6: TCN_FUSION Enhanced + TAPE Three-Component
Architecture: TCN + Fusion
Results root: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results
Working dir: /content/tcn_tape_vectorized_version_clean
Covariance Features: Yes
🎯 REWARD SYSTEM: TAPE (Three-Component v3)
   Profile: BalancedGrowth
   Daily: Base + DSR/PBRS + Turnover_Proximity
   Terminal: mode=signed | baseline=0.20 | scalar=10.0 (clipped ±10.0)
   Gate A: enabled (Sharpe ≤ 0.00 or MDD ≥ 25.0% -> force non-positive terminal bonus)
   Neutral Band: enabled (±0.020 around baseline)
   🔄 Profile Manager: disabled (static profile only)
🎲 Experiment Seed: 6042 (Base: 42, Offset: 6000)
✅ Features: Enhanced

      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00001_shp0p772_actor.weights.h5 (Sharpe=0.772, MDD=18.23%)


      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00008_shp1p039_actor.weights.h5 (Sharpe=1.039, MDD=10.43%)


      💾 Sharpe-threshold checkpoint saved: /content/tcn_tape_vectorized_version_clean/tcn_fusion_results/high_watermark_checkpoints/exp6_tape_hw_ep00012_shp0p934_actor.weights.h5 (Sharpe=0.934, MDD=10.12%)


🔄 Update 10/348 | Step 10,080/500,000 | Episode 12 | Time: 387.8s
   📊 Metrics: Return=+46.21% | Sharpe=0.934 | DD=10.12% | Turnover=36.99%
   🎚️ Intra-Step TAPE: potential=0.6948 | delta_reward=+0.0005
   🎯 Profile: BalancedGrowth
   🧠 Training: actor_loss=0.2004 | critic_loss=0.9248 | mean_adv=0.0000
   🧮 Loss Detail: critic_scaled=0.4624 | risk_aux_total=0.1062 | sharpe_proxy=0.0000 | sharpe_loss=0.0000 | mvo_loss=0.0000 | cvar_proxy=1.7696 | cvar_loss=0.1062 | cvar_coef=0.0600
   ⚙️ Optimizer: actor_lr=0.000030 | critic_lr=0.000150 | target_kl=0.0200 | rollout=1008 | batch_size=252 | gamma=0.9900 | gae_lambda=0.9200
   🔬 Alpha Diversity: mean=6.16 | std=0.86 | range=[3.83, 9.78]


...


📚 TURNOVER CURRICULUM UPDATE at 30,240 steps:
   Turnover penalty scalar: 1.25


🔄 Update 30/348 | Step 30,240/500,000 | Episode 40 | Time: 1167.1s
   📊 Metrics: Return=-31.19% | Sharpe=-0.350 | DD=53.41% | Turnover=80.18%
   🎯 Profile: BalancedGrowth
   🧠 Training: actor_loss=0.2228 | critic_loss=0.8949 | mean_adv=0.0000
   🧮 Loss Detail: critic_scaled=0.4475 | risk_aux_total=0.1411 | sharpe_proxy=0.0000 | sharpe_loss=0.0000 | mvo_loss=0.0000 | cvar_proxy=2.3516 | cvar_loss=0.1411 | cvar_coef=0.0600
   ⚙️ Optimizer: actor_lr=0.000030 | critic_lr=0.000150 | target_kl=0.0200 | rollout=1008 | batch_size=252 | gamma=0.9900 | gae_lambda=0.9200
   🔬 Alpha Diversity: mean=0.69 | std=0.11 | range=[0.52, 1.15]
   ⚠️  WARNING: Alpha std < 0.25 after 30 updates. TCN may not be learning asset discrimination.
   🔒 Drawdown λ snapshot=0.004 (peak 0.004, dd 0.00% / trig 16.50%) | terminal=5.000 (peak 5.000) | TAPE=0.1645


🔄 Update 40/348 | Step 40,320/500,000 | Episode 52 | Time: 1555.9s
   📊 Metrics: Return=+2.20% | Sharpe=-0.044 | DD=15.05% | Turnover=82.78%
   🎚️ Intra-Step TAPE: potential=0.1942 | delta_reward=+0.0000
   🎯 Profile: BalancedGrowth
   🧠 Training: actor_loss=0.2371 | critic_loss=0.3929 | mean_adv=0.0000
   🧮 Loss Detail: critic_scaled=0.1964 | risk_aux_total=0.1427 | sharpe_proxy=0.0000 | sharpe_loss=0.0000 | mvo_loss=0.0000 | cvar_proxy=2.3790 | cvar_loss=0.1427 | cvar_coef=0.0600
   ⚙️ Optimizer: actor_lr=0.000030 | critic_lr=0.000150 | target_kl=0.0200 | rollout=1008 | batch_size=252 | gamma=0.9900 | gae_lambda=0.9200
   🔬 Alpha Diversity: mean=0.67 | std=0.15 | range=[0.41, 1.35]
   ⚠️  WARNING: Alpha std < 0.25 after 40 updates. TCN may not be learning asset discrimination.


🔄 Update 50/348 | Step 50,400/500,000 | Episode 64 | Time: 1942.4s
   📊 Metrics: Return=-26.61% | Sharpe=-0.937 | DD=30.56% | Turnover=79.89%
   🎚️ Intra-Step TAPE: potential=0.5805 | delta_reward=-0.0005
   🎯 Profile: BalancedGrowth
   🧠 Training: actor_loss=0.1787 | critic_loss=0.0860 | mean_adv=0.0000
   🧮 Loss Detail: critic_scaled=0.0430 | risk_aux_total=0.1501 | sharpe_proxy=0.0000 | sharpe_loss=0.0000 | mvo_loss=0.0000 | cvar_proxy=2.5016 | cvar_loss=0.1501 | cvar_coef=0.0600
   ⚙️ Optimizer: actor_lr=0.000030 | critic_lr=0.000150 | target_kl=0.0200 | rollout=1008 | batch_size=252 | gamma=0.9900 | gae_lambda=0.9200
   🔬 Alpha Diversity: mean=0.77 | std=0.20 | range=[0.48, 1.85]
   ⚠️  WARNING: Alpha std < 0.25 after 50 updates. TCN may not be learning asset discrimination.


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1) load latest step diagnostics
logs_dir = Path(globals().get("TRAIN_REPO_DIR", "/content/tcn_tape_vectorized_version_clean")) / "tcn_fusion_results" / "logs"
step_csv = sorted(logs_dir.glob("*_step_diagnostics.csv"))[-1]
diag = pd.read_csv(step_csv)
diag["date"] = pd.to_datetime(diag["date"])

# 2) full-date coverage (all visited dates)
train_dates = pd.to_datetime(train_phase1_data.train_df["Date"]).drop_duplicates().sort_values()
seen_dates = diag["date"].dropna().drop_duplicates().sort_values()

print("Visited-date coverage:",
      f"{len(seen_dates)}/{len(train_dates)} = {len(seen_dates)/len(train_dates):.2%}")

# 3) start-date dispersion (episode_step==1 approximates resets)
starts = diag.loc[diag["episode_step"] == 1, "date"].dropna()
print("Num episode starts:", len(starts))
print("Unique start dates:", starts.nunique())

# 4) start distribution across timeline deciles
rank = starts.rank(method="average", pct=True)
bins = pd.cut(rank, bins=np.linspace(0,1,11), include_lowest=True)
print("Start-date decile distribution:")
print(bins.value_counts().sort_index())


## 6) Inspect Latest Training Logs

In [ ]:
TRAIN_RESULTS_ROOT = Path(globals().get("TRAIN_REPO_DIR", "/content/tcn_tape_vectorized_version_clean")) / "tcn_fusion_results"
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / "logs"

episodes_files = sorted(TRAIN_LOGS_DIR.glob("*episodes*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f"No episodes CSV found in {TRAIN_LOGS_DIR}")
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print("Episodes file:", train_episodes_path)
    print("Rows:", len(train_episodes_df))
    display(train_episodes_df.tail(20))

In [ ]:
#train_episodes_df.columns

## 7) Export Results Folder (Optional)
Creates a zip for download from Colab VM.

In [ ]:
from pathlib import Path
import subprocess

EXPORT_RESULTS_ZIP = True
EXPORT_PATH = Path("/content/tcn_tape_vectorized_run2.zip")
ROOT = Path(globals().get("TRAIN_REPO_DIR", "/content/tcn_tape_vectorized_version_clean"))

if EXPORT_RESULTS_ZIP:
    # Core items
    include_paths = [
        ROOT / "tcn_fusion_results",
        ROOT / "data" / "phase1_preparation_artifacts",
        ROOT / "data" / "master_features_NORMALIZED.csv",
        ROOT / "data_exports",  # include all prep exports like phase1_prep_* artifacts
        ROOT / "output_log",
        ROOT / "output_logs",
    ]

    # Also include latest phase1_prep_* files (explicitly, if present)
    data_exports_dir = ROOT / "data_exports"
    if data_exports_dir.exists():
        latest_prep_files = sorted(
            data_exports_dir.glob("phase1_prep_*"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        include_paths.extend(latest_prep_files)

    # De-dup + existence check
    seen = set()
    existing = []
    for p in include_paths:
        p = p.resolve()
        if p.exists() and p not in seen:
            seen.add(p)
            existing.append(p)

    if not existing:
        print("⚠️ Nothing to export.")
    else:
        if EXPORT_PATH.exists():
            EXPORT_PATH.unlink()

        rel_items = [str(p.relative_to(ROOT)) for p in existing if str(p).startswith(str(ROOT))]
        if not rel_items:
            print("⚠️ No export items are under ROOT.")
        else:
            cmd = f"cd {ROOT} && zip -qr {EXPORT_PATH} " + " ".join(f'"{x}"' for x in rel_items)
            subprocess.run(cmd, shell=True, check=True)

            print(f"✅ Created: {EXPORT_PATH}")
            print("Included:")
            for p in rel_items:
                print(" -", p)
else:
    print("ℹ️ EXPORT_RESULTS_ZIP=False")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/tcn_tape_vectorized_run2.zip /content/drive/MyDrive/
print("✅ Copied to Drive: /content/drive/MyDrive/tcn_tape_vectorized_run2.zip")